In [13]:
from utils import readJson, writeJson

In [14]:
dataset = readJson('Dataset/transfermarkt_fbref_dataset.json')
dataset_manc = readJson('Dataset/transfermarkt_fbref_mancanti.json')
#merge giocatori mancanti con dataset originale
for i in range(len(dataset_manc)):
    for j in range(len(dataset)):
        if dataset_manc[i]['stats'] != {} and dataset_manc[i]['id'] == dataset[j]['id']:
            dataset[j] = dataset_manc[i]

leagues = ['seriea', 'premierleague', 'ligue1', 'laliga', 'bundesliga']
transfers = []
for l in leagues:
    transfers+= readJson(f'Dataset/Transfermarkt/Transfers/{l}_2024.json')
#levo rientri dal prestito
transfers = [x for x in transfers if 'Fine prestito' not in x['cost']]
teams = set(x['tm_team'].upper() for x in dataset)

In [26]:
def computeTransferEvent(transf, dataset, teams):
    event = {}
    for play in dataset:
        if transf['player_id'] == play['tm_id']:
            if transf['team_buyer'].upper() not in teams:
                return {}
            if play['tm_team'].upper() == transf['team_buyer'].upper():
                return event
            event['id'] = play['id']
            event['player_name'] = play['name']
            try:
                if play['tm_role'] == 'Portiere':
                    return {}
                
                event['position'] = play['position']
            except:
                #print(play['name'])
                return {}
            
            event['team'] = play['team']
            event['tm_role'] = play['tm_role']
            event['cost'] = transf['cost']

    return event

In [16]:
for t in transfers:
    if t['player_name'] == 'Nicolò Casale':
        print(t)

{'player_name': 'Nicolò Casale', 'player_id': '389837', 'nationality': 'Italia', 'team_buyer': 'BOLOGNA FC', 'team_seller': 'SS Lazio', 'cost': 'Spesa prestito:\n1,50 mln €'}
{'player_name': 'Nicolò Casale', 'player_id': '389837', 'nationality': 'Italia', 'team_buyer': 'Bologna FC', 'team_seller': 'SS LAZIO', 'cost': 'Spesa prestito:\n1,50 mln €'}


In [28]:
transf_events = []
i=0
log_transfer = {}
for transf in transfers:
    event = computeTransferEvent(transf, dataset,teams)
    if event != {}:
        if event['id'] not in log_transfer.keys():
            transf_events.append(event)
            log_transfer[event['id']] = event['team']
        else:
            if event['team'] == log_transfer[event['id']]:
                #print(f"{event['player_name']}: record duplicato")
                continue
            else:
                print(event, log_transfer[event[id]])
len(transf_events)

300